# Nurse Navigation — 911 Rerouting Insights

Exploratory analysis of 911 nurse-navigation calls. Sorts every call into how it was handled, reads the free-text nurse notes with an LLM to recover *why*, and profiles performance by market and over time.

Focus areas driven by the strategy review: market/county performance, trends over time, Washington State, and sizing the divertible (low-acuity) volume.

## 1. Setup

In [ ]:
import os, re, json, glob, warnings
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 300)

plt.rcParams.update({
    "figure.figsize": (11, 5), "figure.dpi": 110, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090", "#0B2545", "#D1495B", "#E0A500", "#8FA0A6"
PALETTE = [TEAL, NAVY, CORAL, GOLD, "#5FD0BD", "#5A6B72"]

In [ ]:
DATA_DIR = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR  = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
SOURCE_FILE = "data_april2026-aug2026.xlsx"
LLM_MODEL   = "databricks-gpt-oss-120b"
SAMPLE_N    = 500
RANDOM_SEED = 42
MIN_MARKET_CALLS = 100

os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
PRIMARY_INPUT = os.path.join(DATA_DIR, SOURCE_FILE)
RESULTS = {}

def keep(df, name):
    RESULTS[name] = df.copy()
    return df

print("Source:", SOURCE_FILE, "| run:", RUN_ID)

## 2. Load and resolve columns

In [ ]:
def clean_col(c):
    c = re.sub(r"[^\w]+", "_", str(c).strip())
    return re.sub(r"_+", "_", c).lower()

def read_any(path, sheet=0):
    low = path.lower()
    if low.endswith((".xlsx", ".xls")): return pd.read_excel(path, sheet_name=sheet)
    if low.endswith(".parquet"):        return pd.read_parquet(path)
    if low.endswith(".tsv"):            return pd.read_csv(path, sep="\t", low_memory=False)
    return pd.read_csv(path, low_memory=False)

raw = read_any(PRIMARY_INPUT)
raw.columns = [clean_col(c) for c in raw.columns]
print(f"{len(raw):,} rows | {raw.shape[1]} columns")
raw.head(3)

In [ ]:
def find_col(df, exact, contains=None):
    norm = lambda x: x.strip("_")
    normalized = {norm(c): c for c in df.columns}
    for c in exact:
        if c in df.columns: return c
        if norm(c) in normalized: return normalized[norm(c)]
    for pat in (contains or []):
        hits = [c for c in df.columns if pat in c]
        if hits: return sorted(hits, key=len)[0]
    return None

COLS = {
    "notes":   find_col(raw, ["nurses_notes","nurse_notes","notes"], ["nurses_note","note","comment"]),
    "date":    find_col(raw, ["transaction_create_date_time_eastern"], ["date_time","create_date"]),
    "nmtara":  find_col(raw, ["transaction_breakout_including_bls_nmtara_breakout","nmtara_response"], ["nmtara","breakout","tara"]),
    "dispo":   find_col(raw, ["transaction_response_names","response_macro","response"], ["response_name","response"]),
    "market":  find_col(raw, ["market_name","client_name","market"], ["market","client"]),
    "episode": find_col(raw, ["external_reference_number","incident_id"], ["external_reference","incident"]),
    "workset": find_col(raw, ["work_set_id"], ["work_set","workset"]),
    "cause":   find_col(raw, ["cause","chief_complaint","protocol"], ["cause","complaint"]),
}
pd.DataFrame({"logical": list(COLS), "resolved": list(COLS.values())})

In [ ]:
NOTES, DATE, NMTARA, DISPO, MARKET, EPISODE, CAUSE = (
    COLS["notes"], COLS["date"], COLS["nmtara"], COLS["dispo"],
    COLS["market"], COLS["episode"], COLS["cause"])

df = raw.copy()
if DATE:  df[DATE] = pd.to_datetime(df[DATE], errors="coerce")
if NOTES: df[NOTES] = df[NOTES].astype("string")

def nmtara_level(x):
    t = str(x)
    m = re.search(r"(?i)n[am]?tara[^0-9]{0,6}(\d)", t)
    if m: return int(m.group(1))
    if re.search(r"(?i)self[- ]?care", t): return np.nan
    m = re.search(r"(?<![0-9])([0-6])(?![0-9])", t)
    return int(m.group(1)) if m else np.nan

df["nmtara_level"] = df[NMTARA].apply(nmtara_level) if NMTARA else np.nan
_dl = df[DISPO].fillna("").str.lower()
df["is_self_care"]  = _dl.str.contains("self", na=False)
df["is_urgent"]     = _dl.str.contains("urgent", na=False)
df["is_virtual"]    = _dl.str.contains("virtual|telehealth|video", regex=True, na=False)
df["is_ambulance"]  = _dl.str.contains("ambulance|bls|als|911", regex=True, na=False)
df["is_amb_override"] = df["nmtara_level"].between(1,5) & df["is_ambulance"]

def bucket(r):
    if r["nmtara_level"] == 6:   return "NMTARA 6 (triage not completed)"
    if r["is_amb_override"]:     return "Ambulance override"
    if r["is_self_care"]:        return "Self-care"
    return "Other"
df["bucket"] = df.apply(bucket, axis=1)
print(f"{len(df):,} calls loaded")
df[["nmtara_level","bucket"]].head()

## 3. Data coverage

In [ ]:
if DATE:
    monthly = df.dropna(subset=[DATE]).set_index(DATE).resample("MS").size()
    fig, ax = plt.subplots()
    monthly.plot(marker="o", color=TEAL, ax=ax, lw=2)
    ax.set_title("Call volume by month")
    ax.set_ylabel("calls"); ax.set_xlabel("")
    for x, y in zip(monthly.index, monthly.values):
        ax.annotate(f"{y:,}", (x, y), textcoords="offset points", xytext=(0,8), ha="center", fontsize=9)
    plt.tight_layout(); plt.show()
    print(f"Coverage: {df[DATE].min():%b %d, %Y} to {df[DATE].max():%b %d, %Y}  |  {len(df):,} calls")
    cov = monthly.rename("calls").to_frame(); cov.index = cov.index.strftime("%Y-%m")
    keep(cov.reset_index().rename(columns={DATE:"month"}), "data_coverage")

## 4. How calls are handled (the four buckets)

In [ ]:
sizes = (df["bucket"].value_counts()
         .rename("calls").to_frame()
         .assign(pct=lambda d: (d["calls"]/len(df)*100).round(1)))
keep(sizes.reset_index().rename(columns={"index":"bucket"}), "bucket_sizes")
sizes

In [ ]:
order = ["Self-care","NMTARA 6 (triage not completed)","Ambulance override","Other"]
present = [b for b in order if b in sizes.index]
colors = {"Self-care":TEAL,"NMTARA 6 (triage not completed)":GOLD,"Ambulance override":CORAL,"Other":GREY}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13,5), gridspec_kw={"width_ratios":[1,1.1]})
a1.pie(sizes.loc[present,"calls"], labels=None, autopct="%1.1f%%",
       colors=[colors[b] for b in present], startangle=90, wedgeprops=dict(width=0.42, edgecolor="white"))
a1.set_title("Share of all calls")
a1.legend(present, loc="center left", bbox_to_anchor=(-0.15,-0.08), fontsize=9, frameon=False)
b = sizes.loc[present]
a2.barh(range(len(b)), b["calls"], color=[colors[x] for x in present])
a2.set_yticks(range(len(b))); a2.set_yticklabels(present); a2.invert_yaxis()
a2.set_title("Calls per bucket")
for i, v in enumerate(b["calls"]): a2.annotate(f"{v:,}", (v, i), xytext=(5,0), textcoords="offset points", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 5. Trends over time — is the mix shifting?

In [ ]:
if DATE:
    trend = (df.dropna(subset=[DATE]).set_index(DATE)
             .groupby([pd.Grouper(freq="MS"),"bucket"]).size().unstack(fill_value=0))
    trend_pct = trend.div(trend.sum(axis=1), axis=0)*100
    fig, ax = plt.subplots(figsize=(12,5))
    for i, b in enumerate([c for c in order if c in trend_pct.columns]):
        ax.plot(trend_pct.index, trend_pct[b], marker="o", lw=2, label=b, color=colors.get(b, PALETTE[i]))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Bucket mix by month"); ax.set_ylabel("% of calls"); ax.set_xlabel("")
    ax.legend(fontsize=9, frameon=False, ncol=2)
    plt.tight_layout(); plt.show()
    keep(trend_pct.round(1).reset_index().rename(columns={DATE:"month"}), "bucket_trend")
    display(trend_pct.round(1))

## 6. Market / county performance

How well is each market being triaged before it reaches nurse navigation? Lower ambulance %, higher diversion to self-care and urgent care, is the signal.

In [ ]:
if MARKET:
    g = df.groupby(MARKET)
    perf = pd.DataFrame({
        "calls": g.size(),
        "self_care_pct": (g["is_self_care"].mean()*100).round(1),
        "urgent_pct":    (g["is_urgent"].mean()*100).round(1),
        "ambulance_pct": (g["is_ambulance"].mean()*100).round(1),
        "override_pct":  (g["is_amb_override"].mean()*100).round(1),
        "nmtara6_pct":   (g["nmtara_level"].apply(lambda s: s.eq(6).mean())*100).round(1),
    })
    perf["diverted_pct"] = (perf["self_care_pct"] + perf["urgent_pct"]).round(1)
    perf = perf[perf["calls"] >= MIN_MARKET_CALLS].sort_values("diverted_pct", ascending=False)
    keep(perf.reset_index().rename(columns={MARKET:"market"}), "market_performance")
    perf

In [ ]:
if MARKET and len(perf):
    top = perf.sort_values("diverted_pct")
    fig, ax = plt.subplots(figsize=(11, max(4, 0.34*len(top))))
    ax.barh(range(len(top)), top["diverted_pct"], color=TEAL)
    nat = (df["is_self_care"].mean() + df["is_urgent"].mean())*100
    ax.axvline(nat, color=NAVY, ls="--", lw=1.5, label=f"National {nat:.1f}%")
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Diversion rate by market (self-care + urgent care)")
    ax.legend(frameon=False)
    for i, v in enumerate(top["diverted_pct"]): ax.annotate(f"{v:.1f}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
if MARKET and len(perf):
    fig, ax = plt.subplots(figsize=(11, max(4, 0.34*len(perf))))
    srt = perf.sort_values("ambulance_pct", ascending=False)
    ax.barh(range(len(srt)), srt["ambulance_pct"], color=CORAL)
    ax.set_yticks(range(len(srt))); ax.set_yticklabels(srt.index, fontsize=9); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Ambulance rate by market (lower is better)")
    for i, v in enumerate(srt["ambulance_pct"]): ax.annotate(f"{v:.1f}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

## 7. Washington State focus

Washington has enough nurse-navigation coverage to reach roughly half the state, making it the reference market for the Medicaid conversation.

In [ ]:
if MARKET:
    wa_mask = df[MARKET].fillna("").str.contains("WA|washington|seattle|spokane|snohomish|clark|king county", case=False, regex=True)
    wa = df[wa_mask]
    print(f"Washington calls: {len(wa):,}  ({len(wa)/len(df)*100:.1f}% of all calls)")
    if len(wa):
        wa_sizes = (wa["bucket"].value_counts().rename("calls").to_frame()
                    .assign(pct=lambda d: (d["calls"]/len(wa)*100).round(1)))
        display(wa_sizes)
        wa_div = (wa["is_self_care"].mean()+wa["is_urgent"].mean())*100
        nat_div = (df["is_self_care"].mean()+df["is_urgent"].mean())*100
        fig, ax = plt.subplots(figsize=(7,4))
        ax.bar(["Washington","National"], [wa_div, nat_div], color=[TEAL, GREY])
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_title("Diversion rate: Washington vs national")
        for i,v in enumerate([wa_div,nat_div]): ax.annotate(f"{v:.1f}%", (i,v), xytext=(0,5), textcoords="offset points", ha="center")
        plt.tight_layout(); plt.show()
        keep(wa_sizes.reset_index().rename(columns={"index":"bucket"}), "washington_buckets")

In [ ]:
if MARKET and len(wa):
    wg = wa.groupby(MARKET)
    wa_markets = pd.DataFrame({
        "calls": wg.size(),
        "diverted_pct": ((wg["is_self_care"].mean()+wg["is_urgent"].mean())*100).round(1),
        "ambulance_pct": (wg["is_ambulance"].mean()*100).round(1),
    })
    wa_markets = wa_markets[wa_markets["calls"] >= 30].sort_values("diverted_pct", ascending=False)
    if len(wa_markets):
        keep(wa_markets.reset_index().rename(columns={MARKET:"market"}), "washington_markets")
        display(wa_markets)

## 8. Sizing the divertible (low-acuity) volume

The Milliman model turns diverted low-acuity calls into downstream savings. This sizes how many calls were kept out of the ED.

In [ ]:
low_acuity = df["is_self_care"] | df["is_urgent"] | df["is_virtual"]
n_low = int(low_acuity.sum())
per_month = df.dropna(subset=[DATE]).set_index(DATE).resample("MS").apply(lambda x: 1) if DATE else None
months = df[DATE].dt.to_period("M").nunique() if DATE else 1
fig, ax = plt.subplots(figsize=(8,4))
seg = pd.Series({
    "Self-care": int(df["is_self_care"].sum()),
    "Urgent care": int(df["is_urgent"].sum()),
    "Virtual": int(df["is_virtual"].sum()),
})
ax.bar(seg.index, seg.values, color=[TEAL,"#5FD0BD",GOLD])
for i,v in enumerate(seg.values): ax.annotate(f"{v:,}", (i,v), xytext=(0,5), textcoords="offset points", ha="center")
ax.set_title(f"Divertible (low-acuity) calls: {n_low:,} total  (~{n_low/max(months,1):,.0f}/month)")
ax.set_ylabel("calls")
plt.tight_layout(); plt.show()
keep(seg.rename("calls").to_frame().reset_index().rename(columns={"index":"low_acuity_type"}), "divertible_volume")
print(f"Low-acuity share of all calls: {n_low/len(df)*100:.1f}%")

## 9. LLM extraction — recovering the *why* from nurse notes

The disposition code says what happened; the free-text note says why. A sample of notes for the three focus buckets is read by the model, which returns only facts stated in the note, each with a supporting quote.

In [ ]:
from openai import OpenAI

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN", "")
)
client = OpenAI(api_key=DATABRICKS_TOKEN,
                base_url="https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints")

def llm_call(system_prompt, user_prompt, max_tokens=2500):
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}],
        temperature=0.1, max_tokens=max_tokens)
    content = resp.choices[0].message.content
    if isinstance(content, list):
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                return item.get("text","")
        return json.dumps(content)
    return content

In [ ]:
NAV_PROMPT = """
You are a clinical note information extraction assistant for a nurse navigation program.
Extract only facts stated in the "Nurse Notes". Do not guess. Handle negations.
For every boolean set to true you MUST provide a verbatim evidence quote.

OUTPUT valid JSON only, all keys present, booleans never null:
{
  "care_setting_actual": {"setting": "emergency_department | urgent_care | virtual_care | primary_care | home_no_care | unknown", "patient_stayed_home": boolean},
  "transport_mode": {"mode": "ambulance_als | ambulance_bls | rideshare | self_or_family | none | unknown", "arranged_by_program": boolean, "ride_initiated_by": "patient_or_family | program_or_nurse | unknown"},
  "decision_driver": {"patient_refused_recommendation": boolean, "patient_requested_ambulance_or_ed": boolean, "no_provider_available": boolean, "facility_closed_or_after_hours": boolean, "no_appointment_available": boolean, "mobility_or_transport_barrier": boolean, "device_or_procedure_need": boolean, "clinical_escalation_by_nurse": boolean, "insurance_or_cost_barrier": boolean, "language_or_communication_barrier": boolean},
  "triage_incomplete_reason": {"patient_refused_triage": boolean, "call_disconnected_or_technical": boolean, "patient_unable_to_participate": boolean, "protocol_exclusion": boolean, "caller_was_not_patient": boolean},
  "documentation": {"reason_documented": boolean},
  "evidence": [{"field": string, "value": boolean, "quote": string}]
}
Ride: patient_or_family only if the note names the patient or family; if it only says a ride was ordered, use unknown (treated as program-arranged).
Now extract from the following Nurse Notes."""

DRIVERS = ["patient_refused_recommendation","patient_requested_ambulance_or_ed","no_provider_available","facility_closed_or_after_hours","no_appointment_available","mobility_or_transport_barrier","device_or_procedure_need","clinical_escalation_by_nurse","insurance_or_cost_barrier","language_or_communication_barrier"]
TRIAGE = ["patient_refused_triage","call_disconnected_or_technical","patient_unable_to_participate","protocol_exclusion","caller_was_not_patient"]

LABELS = {
 "patient_refused_triage":"Patient would not answer the triage questions (typically wanted the ER)",
 "call_disconnected_or_technical":"The call dropped or hit a technical problem before triage finished",
 "patient_unable_to_participate":"Patient could not take part (confused, unresponsive, or intoxicated)",
 "protocol_exclusion":"The protocol did not allow triage for this type of call",
 "caller_was_not_patient":"The caller was someone other than the patient",
 "clinical_escalation_by_nurse":"The nurse escalated based on clinical judgement",
 "mobility_or_transport_barrier":"The patient could not get there another way (bedbound, no transport)",
 "patient_requested_ambulance_or_ed":"The patient asked for an ambulance or the ER",
 "patient_refused_recommendation":"The patient declined the recommended lower-acuity option",
 "no_provider_available":"No provider or facility was available",
 "facility_closed_or_after_hours":"The facility was closed or it was after hours",
 "no_appointment_available":"No appointment was available",
 "device_or_procedure_need":"The patient needed a device or procedure (for example a catheter)",
 "insurance_or_cost_barrier":"Insurance or cost was a barrier",
 "language_or_communication_barrier":"A language or communication barrier",
}
plabel = lambda k: LABELS.get(k, k.replace("_"," ").capitalize())

In [ ]:
focus = df[df["bucket"] != "Other"].copy()
if NOTES:
    focus = focus[focus[NOTES].fillna("").str.len() >= 20]
per = SAMPLE_N // 3
parts = [g.sample(min(len(g), per), random_state=RANDOM_SEED) for _, g in focus.groupby("bucket")]
sample = pd.concat(parts).reset_index(drop=True)
print(f"Sampling {len(sample):,} calls for extraction")
sample["bucket"].value_counts()

In [ ]:
def valid_json(raw):
    try: obj = json.loads(raw)
    except Exception: return None
    for sec in ("care_setting_actual","transport_mode","decision_driver","triage_incomplete_reason"):
        if sec not in obj: return None
    return obj

rows = []
for _, r in sample.iterrows():
    prompt = f"bucket: {r['bucket']}\ncause: {r.get(CAUSE,'')}\n\nNurse Notes:\n{r.get(NOTES,'')}"
    try:
        obj = valid_json(llm_call(NAV_PROMPT, prompt))
    except Exception:
        obj = None
    rows.append({"idx": r.name, "bucket": r["bucket"], "cause": r.get(CAUSE,""),
                 "notes": str(r.get(NOTES,""))[:400], "ok": obj is not None, "obj": obj})
ext = pd.DataFrame(rows)
print(f"Valid extractions: {ext['ok'].mean()*100:.1f}%  ({ext['ok'].sum()} of {len(ext)})")

In [ ]:
v = ext[ext["ok"]].copy()
for s, keys in [("decision_driver", DRIVERS), ("triage_incomplete_reason", TRIAGE)]:
    for k in keys:
        v[f"{s}.{k}"] = v["obj"].apply(lambda o: bool(o.get(s, {}).get(k, False)))
v["setting"]   = v["obj"].apply(lambda o: o.get("care_setting_actual",{}).get("setting","unknown"))
v["stayed_home"] = v["obj"].apply(lambda o: bool(o.get("care_setting_actual",{}).get("patient_stayed_home",False)))
v["mode"]      = v["obj"].apply(lambda o: o.get("transport_mode",{}).get("mode","unknown"))
v["by_program"]= v["obj"].apply(lambda o: bool(o.get("transport_mode",{}).get("arranged_by_program",False)))
v["ride_by"]   = v["obj"].apply(lambda o: o.get("transport_mode",{}).get("ride_initiated_by","unknown"))
len(v)

## 10. Self-care: was it really self-care?

True self-care means the program provided neither care nor transport. A patient- or family-arranged ride still counts; a program-arranged or unattributed ride does not.

In [ ]:
def true_self_care(r):
    if r["stayed_home"]: return True
    if r["by_program"]:  return False
    return r["ride_by"] == "patient_or_family"

sc = v[v["bucket"]=="Self-care"].copy()
if len(sc):
    sc["class"] = sc.apply(lambda r:
        "Stayed home (self-care)" if r["stayed_home"] else
        ("Patient/family ride (self-care)" if (r["ride_by"]=="patient_or_family" and not r["by_program"]) else
         ("Program/nurse ride (not self-care)" if (r["by_program"] or r["ride_by"]=="program_or_nurse") else
          "Ride not attributed (not self-care)")), axis=1)
    scb = (sc["class"].value_counts(normalize=True).mul(100).round(1).rename("pct_of_self_care").to_frame())
    scb["calls"] = sc["class"].value_counts()
    keep(scb.reset_index().rename(columns={"index":"category"}), "self_care_by_ride")
    fig, ax = plt.subplots(figsize=(10,4))
    o = scb.sort_values("pct_of_self_care")
    cc = [TEAL if "self-care)" in i and "not" not in i else CORAL for i in o.index]
    ax.barh(range(len(o)), o["pct_of_self_care"], color=cc)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Self-care calls by who arranged the ride")
    for i,val in enumerate(o["pct_of_self_care"]): ax.annotate(f"{val}%", (val,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    tsc = sc.apply(true_self_care, axis=1).mean()*100
    print(f"True self-care: {tsc:.1f}%   Mislabeled: {100-tsc:.1f}%")
    display(scb)

## 11. Why triage was not completed (NMTARA 6)

In [ ]:
def reason_table(frame, section, keys, pct_label):
    kk = [k for k in keys if f"{section}.{k}" in frame.columns]
    out = pd.DataFrame({
        "reason":[k.replace("_"," ").capitalize() for k in kk],
        "what it means":[plabel(k) for k in kk],
        pct_label:[round(frame[f"{section}.{k}"].mean()*100,1) for k in kk],
    }).sort_values(pct_label, ascending=False).reset_index(drop=True)
    return out

n6 = v[v["bucket"]=="NMTARA 6 (triage not completed)"]
if len(n6):
    n6t = reason_table(n6, "triage_incomplete_reason", TRIAGE, "% of calls read")
    keep(n6t, "nmtara6_reasons")
    fig, ax = plt.subplots(figsize=(10,4))
    o = n6t.sort_values("% of calls read")
    ax.barh(range(len(o)), o["% of calls read"], color=GOLD)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["reason"], fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Why triage was not completed")
    for i,val in enumerate(o["% of calls read"]): ax.annotate(f"{val}%", (val,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    display(n6t)

## 12. Why an ambulance was sent anyway (overrides)

In [ ]:
ov = v[v["bucket"]=="Ambulance override"]
if len(ov):
    ovt = reason_table(ov, "decision_driver", DRIVERS, "% of override calls")
    ovt = ovt[ovt["% of override calls"] > 0]
    keep(ovt, "override_reasons")
    fig, ax = plt.subplots(figsize=(10,4.5))
    o = ovt.sort_values("% of override calls")
    ax.barh(range(len(o)), o["% of override calls"], color=CORAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["reason"], fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Why an ambulance was sent (reasons can co-occur)")
    for i,val in enumerate(o["% of override calls"]): ax.annotate(f"{val}%", (val,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    display(ovt)

In [ ]:
if len(ov):
    groups = {
        "Nurse / clinical":["clinical_escalation_by_nurse"],
        "Patient":["patient_requested_ambulance_or_ed","patient_refused_recommendation"],
        "Access / logistics":["no_provider_available","facility_closed_or_after_hours","no_appointment_available","insurance_or_cost_barrier"],
        "Mobility / device":["mobility_or_transport_barrier","device_or_procedure_need"],
    }
    def any_of(frame, keys):
        cols=[f"decision_driver.{k}" for k in keys if f"decision_driver.{k}" in frame.columns]
        return frame[cols].any(axis=1) if cols else pd.Series(False, index=frame.index)
    gsplit = pd.DataFrame({"driver_group":list(groups),
        "% of override calls":[round(any_of(ov,ks).mean()*100,1) for ks in groups.values()]}).sort_values("% of override calls", ascending=False)
    keep(gsplit, "override_driver_split")
    fig, ax = plt.subplots(figsize=(9,4))
    o = gsplit.sort_values("% of override calls")
    ax.barh(range(len(o)), o["% of override calls"], color=NAVY)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["driver_group"], fontsize=10)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Override drivers: who or what drove the decision")
    for i,val in enumerate(o["% of override calls"]): ax.annotate(f"{val}%", (val,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    display(gsplit)

## 13. Override opportunities — what could be diverted

Access and appointment barriers may be divertible to urgent care or telehealth; mobility barriers point to a transport solution; clinical escalations are the nurse's judgement and stay.

In [ ]:
if len(ov):
    opp = gsplit.copy()
    opp["opportunity"] = opp["driver_group"].map({
        "Access / logistics":"Divertible to UC / telehealth",
        "Mobility / device":"Transport solution",
        "Patient":"Patient education / expectation",
        "Nurse / clinical":"Clinical judgement (stays)",
    })
    keep(opp, "override_opportunities")
    display(opp)

In [ ]:
if len(ov) and CAUSE:
    top_cause = (ov["cause"].replace("","Not stated").value_counts()
                 .rename("calls").to_frame()
                 .assign(**{"% of overrides": lambda d:(d["calls"]/len(ov)*100).round(1)}).head(10))
    keep(top_cause.reset_index().rename(columns={"index":"chief_complaint"}), "override_top_complaints")
    fig, ax = plt.subplots(figsize=(10,4.5))
    o = top_cause.sort_values("calls")
    ax.barh(range(len(o)), o["calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.set_title("Top chief complaints behind overrides")
    for i,val in enumerate(o["calls"]): ax.annotate(f"{val}", (val,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    display(top_cause)

## 14. Mobility drivers

In [ ]:
MOBILITY = {
 "Wheelchair":["wheelchair","w/c"], "Bedbound":["bedbound","bed bound","bed-bound"],
 "Cannot ambulate":["cannot ambulate","unable to ambulate","non-ambulatory","unable to walk"],
 "Stairs":["stairs","staircase","second floor","upstairs"],
 "Bariatric / lift":["bariatric","lift assist","lift-assist","two person"],
 "No transport":["no transport","no ride","no car","no one to drive","no way to get"],
}
if len(ov) and "decision_driver.mobility_or_transport_barrier" in ov.columns and NOTES:
    mob = ov[ov["decision_driver.mobility_or_transport_barrier"]]
    if len(mob):
        text = mob["notes"].fillna("").str.lower()
        md_rows = [{"mobility_driver":lab,"calls":int(text.apply(lambda t: any(x in t for x in terms)).sum())} for lab,terms in MOBILITY.items()]
        mobd = pd.DataFrame(md_rows).sort_values("calls", ascending=False).reset_index(drop=True)
        mobd["% of mobility overrides"] = (mobd["calls"]/len(mob)*100).round(1)
        keep(mobd, "mobility_drivers")
        fig, ax = plt.subplots(figsize=(9,4))
        o = mobd.sort_values("calls")
        ax.barh(range(len(o)), o["calls"], color="#5FD0BD")
        ax.set_yticks(range(len(o))); ax.set_yticklabels(o["mobility_driver"], fontsize=9)
        ax.set_title("What drives mobility-barrier overrides")
        for i,val in enumerate(o["calls"]): ax.annotate(f"{val}", (val,i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
        plt.tight_layout(); plt.show()
        display(mobd)

## 15. Evidence sample

Every extracted flag is backed by a verbatim quote from the note, so any finding can be checked.

In [ ]:
ev_rows = []
for _, r in v.head(30).iterrows():
    for e in (r["obj"].get("evidence") or []):
        ev_rows.append({"bucket":r["bucket"],"field":e.get("field"),"value":e.get("value"),
                        "quote":str(e.get("quote",""))[:200]})
evidence = pd.DataFrame(ev_rows)
keep(evidence, "evidence_sample")
evidence.head(25)

## 16. Write the workbook

In [ ]:
def sanitize(d):
    o = d.copy(); o.columns=[str(c) for c in o.columns]
    for c in o.columns:
        if o[c].dtype == object:
            o[c] = o[c].apply(lambda x: "" if x is None or (isinstance(x,float) and pd.isna(x)) else str(x))
    return o

TABS = [
    ("bucket_sizes","Bucket Sizes"), ("bucket_trend","Bucket Trend"), ("data_coverage","Data Coverage"),
    ("market_performance","Market Performance"), ("washington_buckets","Washington Buckets"),
    ("washington_markets","Washington Markets"), ("divertible_volume","Divertible Volume"),
    ("self_care_by_ride","Self-Care by Ride"), ("nmtara6_reasons","NMTARA 6 Reasons"),
    ("override_reasons","Override Reasons"), ("override_driver_split","Override Driver Split"),
    ("override_opportunities","Override Opportunities"), ("override_top_complaints","Override Top Complaints"),
    ("mobility_drivers","Mobility Drivers"), ("evidence_sample","Evidence Sample"),
]
xlsx_path = os.path.join(OUT_DIR, f"Nurse_Nav_Insights_{RUN_ID}.xlsx")
try:
    import xlsxwriter; engine="xlsxwriter"
except ImportError:
    engine="openpyxl"
with pd.ExcelWriter(xlsx_path, engine=engine) as w:
    pd.DataFrame({"Nurse Navigation - 911 Rerouting Insights":[
        f"Run {RUN_ID}", f"Source: {SOURCE_FILE}", f"Calls: {len(df):,}",
        "Tabs cover call handling, trends, market performance, Washington State,",
        "divertible volume, and the LLM-abstracted reasons behind each decision.",
    ]}).to_excel(w, sheet_name="Start Here", index=False)
    written = 0
    for stem, tab in TABS:
        d = RESULTS.get(stem)
        if d is None or len(d)==0: 
            print("skip", tab); continue
        sanitize(d).to_excel(w, sheet_name=tab[:31], index=False)
        written += 1; print("added", tab)
for old in glob.glob(os.path.join(OUT_DIR, "Nurse_Nav_Insights_*.xlsx")):
    if os.path.abspath(old) != os.path.abspath(xlsx_path):
        try: os.remove(old)
        except Exception: pass
print(f"\nWorkbook: {os.path.basename(xlsx_path)}  ({written+1} tabs)")